# Program for running back-classification of NACE on test data
1. Read in data
2. Classify 1-to-1
3. Identify problem manual groups
4. Classify using rules
5. Classify using machine learning model

To do:
- add in flag variable
- Move things to config

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import s3fs

import mlflow

import nltk
from nltk.corpus import stopwords

In [ ]:
from utils.one_to_one import get_1to1_dict, get_problem_NACE
from utils.rules_classification import norwegian_rules, classify_rules
from utils.ml_classification import get_label, get_possibles
from utils.text_processing import preprocess


In [ ]:
# Set path for reading data
bucket = "projet-aiml4os-wp10"
bucket_url = f"https://minio.lab.sspcloud.fr/{bucket}"
test_path = f"{bucket_url}/Cluster5/test_doublenace_2026-03-23.parquet"
conversion_path = f"{bucket_url}/Cluster5/NACE2.1-NACE2_Table_V1.05.xlsx"

In [ ]:
# Set model id and paths
model_id = "m-8d82ed413d9f48febf123e4f4216c29f"
model_path = f"s3://{bucket}/Cluster5/mlflow_artifacts/models/{model_id}/artifacts"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]

In [ ]:
# Set seed
seed = 42

### Read in data

In [ ]:
test = pd.read_parquet(test_path)
print(f'Number of observations in test: {test.shape[0]}')

In [ ]:
# Drop NACE 2 code for demonstartion purposes on how this can be predicted
test = test.drop(columns="nace_20_code")

### Classify 1-to-1

In [ ]:
# Create dictionary of additional 1-to-1 relationships which are country/regional level
country_dict = {
        "00.00": "00.00",
        "01.41": "01.41",
        "01.63": "01.63",
        "10.62": "10.62",
        "11.07": "11.07",
        "25.21": "25.21",
        "26.70": "26.70",
        "26.60": "32.50",
        "55.90": "55.90",
        "58.19": "58.19",
        "77.22": "77.29",
        "78.20": "78.20",
    }

# Get standard dictionary of 1-to-1 relationships
one_to_one_dict = get_1to1_dict(conversion_path, country_dict) 

In [ ]:
# Apply to data
test["nace_20_code"] = test.nace_21_code.map(one_to_one_dict)
test["nace_20_source"] = np.where(test.nace_20_code.notna(), "one-to-one", pd.NA)

n_1to1 = test.nace_20_code.notna().sum()
print(f'Number of observations classifed using 1-to-1: {n_1to1}')

### Identify problem NACE for manual coding

In [ ]:
problem_nace = get_problem_NACE(conversion_path)

In [ ]:
mask_problem = test.nace_21_code.isin(problem_nace)
test.loc[mask_problem, "nace_20_code"] = "manual"
test.loc[mask_problem, "nace_20_source"] = "manual"

In [ ]:
print(f'Number of observations in problem groups: {mask_problem.sum()}')

### Classify using rules


In [ ]:
# Create text variable
test["company_name"] = test["company_name"].fillna("")
test["company_activity"] = test["company_activity"].fillna("")
test["company_name_description"] = test["company_name"] + test["company_activity"]

In [ ]:
# Classify
nace_20_rules = classify_rules(test, keyword_rules=norwegian_rules, company_nace21="nace_21_code", 
        company_text="company_name_description")
test["nace_20_code"] = test["nace_20_code"].fillna(nace_20_rules)
test["nace_20_source"] = np.where(nace_20_rules.notna(), "rules", test.nace_20_source)

In [ ]:
print(f'Number of observations with rules classification: {nace_20_rules.notna().sum()}')

### Classify using machine learning model


In [ ]:
print(f'Number of observations requiring classification with ML model: {test.nace_20_code.isna().sum()}')

**Preprocess test**

In [ ]:
# Get stop words
nltk.download('stopwords') # one time to use locally
norwegian_stopwords = set(stopwords.words('norwegian'))|{"as"}

# Create text variable
test["company_name"] = test["company_name"].fillna("")
test["company_activity"] = test["company_activity"].fillna("")
test["company_text"] = test["company_name"] + " " + test["company_activity"]

# Preprocess and add in nace 2.1
#text_variable, stopwords, nace_21_variable = "nace_21_code"
test["company_text_processed"] = preprocess(test, 
                                             text_variable="company_text", 
                                             stopwords = norwegian_stopwords, 
                                             nace_21_variable = "nace_21_code"
                                            )

**Fetch model**

In [ ]:
model = mlflow.sklearn.load_model(model_path)

In [ ]:
# Predict using model (prediction matrix)
pred_mat = model.predict_proba(test.company_text_processed)

In [ ]:
# Select from only the possible NACE 2.0 codes, based on NACE rev. 2.1
all_labels = model.classes_
possibles = get_possibles(test, company_nace21 = "nace_21_code", conversion_path=conversion_path)
labels_mat = [[l for l in all_labels] for _ in range(test.shape[0])]

# Predict NACE: check this function! - dont need labels_svc
ml_labels = get_label(labels_mat, pred_mat, possibles, all_labels, choose_type = "random", seed = seed)

test["nace_20_source"] = np.where(test.nace_20_code.isna(), "ml", test.nace_20_source)

In [ ]:
test["nace_20_code"] = test["nace_20_code"].fillna(pd.Series(ml_labels[0]))

In [ ]:
print(f"Number still missing nace 2.0: {test.nace_20_code.isna().sum()}")